In [ ]:
import os

KEEP_RUNTIME = True  # False restores the worker's default 10-minute idle release.
if KEEP_RUNTIME:
    os.environ['IDLE_MINUTES'] = '100000'
else:
    os.environ.pop('IDLE_MINUTES', None)

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import re
import subprocess
from pathlib import Path

QUEUE_REF = '100bb43a7c976a5d4ab751002441964fdb8b2724'  # immutable queue implementation SHA
DRIVE_ROOT = '/content/drive/MyDrive/t16_colab'
WORKER_READY_REF = None
assert re.fullmatch(r'[0-9a-f]{40}', QUEUE_REF) and set(QUEUE_REF) != {'0'}
bootstrap = Path('/content/colab-queue')
bootstrap.mkdir(parents=True, exist_ok=True)  # local runtime storage only
worker = bootstrap / 'worker.sh'
download = bootstrap / 'worker.sh.download'
url = f'https://raw.githubusercontent.com/josenunezwork/snake-dqn/{QUEUE_REF}/research/colab_queue/worker.sh'
subprocess.run(['curl', '-fsSL', '--retry', '3', '--max-time', '120', url, '-o', str(download)], check=True)
download.replace(worker)
WORKER_READY_REF = QUEUE_REF


In [ ]:
import os
import subprocess
import sys
import uuid
from pathlib import Path

assert WORKER_READY_REF == QUEUE_REF, 'Run the pinned fetch cell successfully first'
env = os.environ.copy()
env['QUEUE_REF'] = QUEUE_REF
env['DRIVE_ROOT'] = DRIVE_ROOT
env['COLAB_PYTHON'] = sys.executable
worker_id = f'notebook-{uuid.uuid4().hex}'
log_path = Path('/content/colab-queue/worker.log')
with log_path.open('ab', buffering=0) as log:
    proc = subprocess.Popen(
        ['nohup', 'bash', '/content/colab-queue/worker.sh', '--worker-id', worker_id],
        cwd='/content/colab-queue', env=env, stdin=subprocess.DEVNULL, stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
    )
print(f'queue worker pid={proc.pid}; local log={log_path}; Drive root={env["DRIVE_ROOT"]}')


In [ ]:
# Keep-alive: leave this LAST cell running after Runtime > Run all.
import json
from datetime import datetime, timezone

heartbeat_path = Path(DRIVE_ROOT) / 'queue' / 'heartbeats' / f'{worker_id}.json'
while True:
    alive = proc.poll() is None
    try:
        with log_path.open('rb') as log:
            log.seek(0, 2)
            log.seek(max(0, log.tell() - 8192))
            lines = log.read().decode('utf-8', errors='replace').splitlines()
        last_line = lines[-1] if lines else '(empty)'
    except OSError as exc:
        last_line = f'(log unavailable: {exc})'
    try:
        heartbeat = json.loads(heartbeat_path.read_text())
        current_job = heartbeat.get('job_id') or '(idle)'
    except (OSError, ValueError) as exc:
        current_job = f'(heartbeat unavailable: {exc})'
    print(f'{datetime.now(timezone.utc).isoformat()} worker alive: {"yes" if alive else "no"}; '
          f'current queue job: {current_job}; last worker.log line: {last_line}', flush=True)
    if not alive:
        break
    try:
        proc.wait(timeout=600)  # Wake immediately on exit, including idle release.
    except subprocess.TimeoutExpired:
        pass
